In [1]:
# 1) Mount Drive (if not yet)
# -------------------------
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:

import os, shutil, datetime, warnings, gc, time, joblib
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image

from joblib import Parallel, delayed
from skimage.feature import hog

from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import label_binarize, FunctionTransformer
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, accuracy_score, f1_score)


In [3]:
# ---------------- PATHS ----------------
DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks"
SPLIT80_DIR = f"{DRIVE_ROOT}/emotion/emotion_preprocessed_split80_20_aug4000"  # <-- your 80/20 aug path
TEST_DIR    = f"{DRIVE_ROOT}/emotion/emotion_preprocessed/test"

LOCAL_TRAIN = "/content/data/train"
LOCAL_VAL   = "/content/data/val"
LOCAL_TEST  = "/content/data/test"

OUT_DIR = f"{DRIVE_ROOT}/NB_HOG_80_20_FAST_TUNED_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(OUT_DIR, exist_ok=True)

def fresh_copy(src, dst):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

# Local copy for faster IO
fresh_copy(os.path.join(SPLIT80_DIR, "train"), LOCAL_TRAIN)
fresh_copy(os.path.join(SPLIT80_DIR, "val"),   LOCAL_VAL)
fresh_copy(TEST_DIR, LOCAL_TEST)
print("✅ Copied 80/20 (aug) train/val + test to Colab local.")
print("OUT_DIR:", OUT_DIR)


✅ Copied 80/20 (aug) train/val + test to Colab local.
OUT_DIR: /content/drive/MyDrive/Colab Notebooks/NB_HOG_80_20_FAST_TUNED_20250930_130056


In [4]:

# ---------------- CONFIG ----------------
RANDOM_STATE = 42
IMG_SIZE = (48, 48)

# Light & fast HOG (works well on faces)
HOG_PARAMS = dict(
    orientations=8,
    pixels_per_cell=(10,10),
    cells_per_block=(2,2),
    block_norm="L2-Hys",
    transform_sqrt=True,
    feature_vector=True
)

# Tuning budget (fast but effective)
TUNE_CV   = 2     # keep small for speed
NITER_GNB = 10    # GaussianNB tuning iterations
NITER_CNB = 8     # ComplementNB tuning iterations


In [5]:
# ---------------- UTILS ----------------
def list_classes_from_dir(d):
    return sorted([x for x in os.listdir(d) if os.path.isdir(os.path.join(d, x))])

def load_paths_and_labels(root_dir, class_names):
    X_paths, y = [], []
    for c in class_names:
        d = os.path.join(root_dir, c)
        ps = sorted(glob(os.path.join(d, "*")))
        X_paths.extend(ps)
        y.extend([class_names.index(c)]*len(ps))
    return X_paths, np.array(y, dtype=np.int64)

def read_gray(path):
    return np.asarray(Image.open(path).convert("L").resize(IMG_SIZE), dtype=np.uint8)

def hog_one(path):
    return hog(read_gray(path), **HOG_PARAMS)

def compute_hog(paths, n_jobs=-1, batch_size=256):
    feats = Parallel(n_jobs=n_jobs, prefer="threads", batch_size=batch_size)(
        delayed(hog_one)(p) for p in paths
    )
    return np.vstack(feats)

def cache_path(name): return os.path.join(OUT_DIR, name)

def get_or_build_hog(tag, paths):
    cp = cache_path(f"hog_{tag}.npy")
    if os.path.exists(cp):
        print(f"⚡ Loading cached HOG: {cp}")
        return np.load(cp)
    print(f"🛠️ Extracting HOG for {tag} …")
    X = compute_hog(paths, n_jobs=-1)
    np.save(cp, X)
    return X

def save_txt(path, text):
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

def plot_confusion(cm, classes, out_png, title):
    try:
        import seaborn as sns
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt="d", xticklabels=classes, yticklabels=classes)
    except Exception:
        plt.figure(figsize=(8,6))
        plt.imshow(cm, interpolation="nearest"); plt.colorbar()
        ticks = np.arange(len(classes))
        plt.xticks(ticks, classes, rotation=45); plt.yticks(ticks, classes)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i,j]), ha="center", va="center")
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()

def plot_roc_ovr(y_true, y_proba, classes, out_png, title):
    y_bin = label_binarize(y_true, classes=list(range(len(classes))))
    fpr, tpr, roc_auc = {}, {}, {}
    for i in range(len(classes)):
        fpr[i], tpr[i], _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(len(classes))]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(len(classes)):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(classes)
    macro_auc = auc(all_fpr, mean_tpr)
    plt.figure(figsize=(8,6))
    for i in range(len(classes)):
        plt.plot(fpr[i], tpr[i], label=f"{classes[i]} (AUC={roc_auc[i]:.3f})")
    plt.plot(all_fpr, mean_tpr, linestyle="--", label=f"macro (AUC={macro_auc:.3f})")
    plt.plot([0,1],[0,1], linestyle=":", label="chance")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(title); plt.legend(fontsize=8, loc="lower right")
    plt.tight_layout(); plt.savefig(out_png, dpi=150); plt.close()

In [6]:
# ---------------- DATA & FEATURES ----------------
CLASSES = list_classes_from_dir(LOCAL_TRAIN)
print("Classes:", CLASSES)

train_paths, y_train = load_paths_and_labels(LOCAL_TRAIN, CLASSES)
val_paths,   y_val  = load_paths_and_labels(LOCAL_VAL,   CLASSES)
test_paths,  y_test = load_paths_and_labels(LOCAL_TEST,  CLASSES)
print(f"Counts: train={len(train_paths)}, val={len(val_paths)}, test={len(test_paths)}")

t0 = time.time()
X_train = get_or_build_hog("train", train_paths)
X_val   = get_or_build_hog("val",   val_paths)
X_test  = get_or_build_hog("test",  test_paths)
print(f"HOG total time: {time.time()-t0:.1f}s")
print("HOG shapes:", X_train.shape, X_val.shape, X_test.shape)


Classes: ['angry', 'disgusted', 'fearful', 'happy', 'neutral', 'sad', 'surprised']
Counts: train=28000, val=5744, test=7178
🛠️ Extracting HOG for train …
🛠️ Extracting HOG for val …
🛠️ Extracting HOG for test …
HOG total time: 37.0s
HOG shapes: (28000, 288) (5744, 288) (7178, 288)


In [7]:
# ---------------- FAST LIGHT TUNING ----------------
# A) GaussianNB (+ optional PCA)
pipe_gnb = Pipeline(steps=[
    ("pca", PCA(random_state=RANDOM_STATE)),  # n_components tuned incl. None
    ("gnb", GaussianNB())
])
param_dist_gnb = {
    "pca__n_components": [None, 128, 192, 256, 384],
    "gnb__var_smoothing": np.logspace(-12, -3, 10)
}
gnb_search = RandomizedSearchCV(
    estimator=pipe_gnb,
    param_distributions=param_dist_gnb,
    n_iter=NITER_GNB,
    scoring="f1_macro",
    cv=TUNE_CV,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    refit=True
)

In [8]:
# B) ComplementNB needs non-negative "count-like" features
def to_counts(X):
    X = X.astype(np.float32)
    s = X.sum(axis=1, keepdims=True) + 1e-9
    Xn = X / s
    return np.round(Xn * 255.0).astype(np.int32)

pipe_cnb = Pipeline(steps=[
    ("cnt", FunctionTransformer(to_counts, validate=False)),
    ("cnb", ComplementNB())
])
param_dist_cnb = {
    "cnb__alpha": np.logspace(-2, 1, 8)  # 0.01 .. 10
}
cnb_search = RandomizedSearchCV(
    estimator=pipe_cnb,
    param_distributions=param_dist_cnb,
    n_iter=NITER_CNB,
    scoring="f1_macro",
    cv=TUNE_CV,
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    refit=True
)

print("🔎 Tuning GaussianNB …")
t1 = time.time()
gnb_search.fit(X_train, y_train)
print(f"GNB tuning time: {time.time()-t1:.1f}s | best:", gnb_search.best_params_, "| CV f1_macro:", gnb_search.best_score_)

print("🔎 Tuning ComplementNB …")
t2 = time.time()
cnb_search.fit(X_train, y_train)
print(f"CNB tuning time: {time.time()-t2:.1f}s | best:", cnb_search.best_params_, "| CV f1_macro:", cnb_search.best_score_)


🔎 Tuning GaussianNB …
Fitting 2 folds for each of 10 candidates, totalling 20 fits
GNB tuning time: 7.5s | best: {'pca__n_components': 128, 'gnb__var_smoothing': np.float64(1e-07)} | CV f1_macro: 0.3357858132129963
🔎 Tuning ComplementNB …
Fitting 2 folds for each of 8 candidates, totalling 16 fits
CNB tuning time: 4.8s | best: {'cnb__alpha': np.float64(0.01)} | CV f1_macro: 0.26709325455793476


In [9]:

# ---------------- VALIDATION EVAL & PICK BEST ----------------
def eval_on_val(name, model, Xv, yv):
    y_pred = model.predict(Xv)
    acc = accuracy_score(yv, y_pred)
    f1m = f1_score(yv, y_pred, average="macro")
    print(f"VAL {name}: acc={acc:.4f}, f1_macro={f1m:.4f}")
    return acc, f1m, y_pred

acc_g, f1_g, yv_pred_g = eval_on_val("GaussianNB",   gnb_search.best_estimator_, X_val, y_val)
acc_c, f1_c, yv_pred_c = eval_on_val("ComplementNB", cnb_search.best_estimator_, X_val, y_val)

if (f1_c > f1_g) or (f1_c == f1_g and acc_c >= acc_g):
    best_tag = "ComplementNB"
    best_model = cnb_search.best_estimator_
    yv_pred_best = yv_pred_c
else:
    best_tag = "GaussianNB"
    best_model = gnb_search.best_estimator_
    yv_pred_best = yv_pred_g

print(f"✅ Chosen NB variant: {best_tag}")

# Save VAL artifacts
val_report = classification_report(y_val, yv_pred_best, target_names=CLASSES, digits=4)
save_txt(cache_path(f"val_classification_report_{best_tag}.txt"), val_report)
cm_val = confusion_matrix(y_val, yv_pred_best)
plot_confusion(cm_val, CLASSES, cache_path(f"val_confusion_{best_tag}.png"),
               title=f"Confusion Matrix (VAL) – {best_tag}+HOG (tuned)")


VAL GaussianNB: acc=0.3724, f1_macro=0.3162
VAL ComplementNB: acc=0.2897, f1_macro=0.2322
✅ Chosen NB variant: GaussianNB


In [10]:
# ROC (if available)
try:
    y_val_proba = best_model.predict_proba(X_val)
    plot_roc_ovr(y_val, y_val_proba, CLASSES, cache_path(f"val_roc_ovr_{best_tag}.png"),
                 title=f"ROC (OvR) – VAL – {best_tag}+HOG (tuned)")
except Exception:
    pass


In [11]:
# ---------------- FINAL REFIT (TRAIN+VAL) & TEST ----------------
X_trval = np.vstack([X_train, X_val])
y_trval = np.concatenate([y_train, y_val])

from sklearn.base import clone
final_nb = clone(best_model)
print(f"Refitting {best_tag} on TRAIN+VAL …")
t3 = time.time()
final_nb.fit(X_trval, y_trval)
print(f"Refit time: {time.time()-t3:.1f}s")
joblib.dump(final_nb, cache_path(f"{best_tag}_hog_final_trval.joblib"))

# TEST
y_test_pred = final_nb.predict(X_test)
test_report = classification_report(y_test, y_test_pred, target_names=CLASSES, digits=4)
save_txt(cache_path(f"test_classification_report_{best_tag}.txt"), test_report)

cm_test = confusion_matrix(y_test, y_test_pred)
plot_confusion(cm_test, CLASSES, cache_path(f"test_confusion_{best_tag}.png"),
               title=f"Confusion Matrix (TEST) – {best_tag}+HOG (tuned)")

try:
    y_test_proba = final_nb.predict_proba(X_test)
    plot_roc_ovr(y_test, y_test_proba, CLASSES, cache_path(f"test_roc_ovr_{best_tag}.png"),
                 title=f"ROC (OvR) – TEST – {best_tag}+HOG (tuned)")
except Exception:
    pass

print("✅ Saved outputs to:", OUT_DIR)
print("\n==== VAL REPORT (chosen) ====\n", val_report)
print("\n==== TEST REPORT (chosen) ====\n", test_report)

# cleanup
del X_train, X_val, X_test, X_trval, y_trval
gc.collect()

Refitting GaussianNB on TRAIN+VAL …
Refit time: 0.2s
✅ Saved outputs to: /content/drive/MyDrive/Colab Notebooks/NB_HOG_80_20_FAST_TUNED_20250930_130056

==== VAL REPORT (chosen) ====
               precision    recall  f1-score   support

       angry     0.2967    0.2128    0.2478       799
   disgusted     0.0694    0.2273    0.1064        88
     fearful     0.2698    0.1451    0.1887       820
       happy     0.5516    0.5745    0.5628      1443
     neutral     0.3316    0.3927    0.3596       993
         sad     0.3059    0.3582    0.3300       966
   surprised     0.4193    0.4173    0.4183       635

    accuracy                         0.3724      5744
   macro avg     0.3206    0.3326    0.3162      5744
weighted avg     0.3746    0.3724    0.3683      5744


==== TEST REPORT (chosen) ====
               precision    recall  f1-score   support

       angry     0.3034    0.2077    0.2466       958
   disgusted     0.0909    0.2523    0.1337       111
     fearful     0.2687

12655